In [ ]:
import datetime as dt
import os
import sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

import concurrent.futures
import itertools
from multiprocessing import Pool, cpu_count

import dask.array as da
import dask.dataframe as dd
import pandas as pd
import numpy as np
import statsmodels.api as sm
from dotenv import load_dotenv
from mc_postgres_db import models as mc
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session, aliased
from statsmodels.regression.rolling import RollingOLS
from statsmodels.tsa.stattools import adfuller
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

from utils.data import align_series, freq_to_window

## Backtesting Cointegration on N Currencies

In this notebook I will apply my pairs trading strategy against historical data. I will attempt to create this using vectorized functions so that I can iterate quickly against large amounts of historical data (around 1 year or longer). The implementation leverages NumPy's vectorized operations to achieve computational efficiency, enabling rapid backtesting across extended temporal horizons while maintaining statistical rigor. This approach is particularly crucial for cointegration analysis, where the computational complexity of traditional iterative methods scales exponentially with the number of currency pairs and observation periods. By employing vectorized computations, we can efficiently evaluate the statistical significance of cointegration relationships across multiple timeframes and conduct comprehensive robustness checks that are essential for validating the empirical validity of our trading strategy in diverse market conditions.

I will start by setting up my environment to pull historical data.

In [ ]:
load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

engine = create_engine(POSTGRES_URL)

Now, I will define the parameters of my test in terms of the period I would like to iterate over and anything else that is important to the simulation.

In [ ]:
start = dt.datetime(2024, 1, 1)
end = start + dt.timedelta(days=30)
lookback_window = freq_to_window("1m", "7D")
lookback_step = freq_to_window("1m", "1D")

After our parameters are defined, we can pull our historical data.

In [ ]:
# Get the Kraken provider.
with Session(engine) as session:
    stmt = select(mc.Provider).where(mc.Provider.name == "Kraken")
    kraken = session.execute(stmt).scalar_one()

# Get the USD asset.
with Session(engine) as session:
    stmt = select(mc.Asset).where(mc.Asset.name == "USD")
    usd = session.execute(stmt).scalar_one()

# Get all asset pairs from Kraken.
from_asset = aliased(mc.Asset)
to_asset = aliased(mc.Asset)
data = pd.read_sql(
    select(
        mc.ProviderAssetMarket.timestamp,
        mc.ProviderAssetMarket.from_asset_id,
        mc.ProviderAssetMarket.to_asset_id,
        from_asset.name.label("from_asset_name"),
        to_asset.name.label("to_asset_name"),
        mc.ProviderAssetMarket.close,
    )
    .join(to_asset, mc.ProviderAssetMarket.to_asset_id == to_asset.id)
    .join(from_asset, mc.ProviderAssetMarket.from_asset_id == from_asset.id)
    .where(
        mc.ProviderAssetMarket.provider_id == kraken.id,
        mc.ProviderAssetMarket.from_asset_id == usd.id,
        mc.ProviderAssetMarket.timestamp >= start,
        mc.ProviderAssetMarket.timestamp <= end,
    )
    .order_by(mc.ProviderAssetMarket.timestamp),
    engine,
)
display(data)

From these currencies, we then want to find every possible pair combination so that we can check for cointegration. I will first find these combinations and then generate a new data frame that we can perform cointegration on at once.

In [ ]:
# Get the combinations of currencies.
timeframe = pd.date_range(start=start, end=end, freq="1min")
currency_ids = (
    data[["from_asset_id", "to_asset_id"]]
    .drop_duplicates()
    .apply(tuple, axis=1)
    .tolist()
)
currency_id_combinations = list(itertools.combinations(currency_ids, 2))
print(
    f"There are {len(currency_id_combinations)} combinations of currencies in the data."
)


# For very large datasets, you can also use Dask for the cartesian product
def create_calculation_frame_dask(timeframe, currency_combinations):
    """Create calculation frame using Dask for memory efficiency"""

    def process_chunk(chunk_combinations):
        chunk_data = [
            {
                "timestamp": timestamp,
                "from_asset_id_1": from_asset_id[0],
                "to_asset_id_1": from_asset_id[1],
                "from_asset_id_2": to_asset_id[0],
                "to_asset_id_2": to_asset_id[1],
            }
            for timestamp in timeframe
            for from_asset_id, to_asset_id in chunk_combinations
        ]
        chunk_data = pd.DataFrame(chunk_data)
        chunk_data = chunk_data.merge(
            data.rename(
                columns={
                    "from_asset_id": "from_asset_id_1",
                    "to_asset_id": "to_asset_id_1",
                    "from_asset_name": "from_asset_name_1",
                    "to_asset_name": "to_asset_name_1",
                    "close": "close_1",
                }
            ),
            on=["timestamp", "from_asset_id_1", "to_asset_id_1"],
            how="left",
        )
        chunk_data = chunk_data.merge(
            data.rename(
                columns={
                    "from_asset_id": "from_asset_id_2",
                    "to_asset_id": "to_asset_id_2",
                    "from_asset_name": "from_asset_name_2",
                    "to_asset_name": "to_asset_name_2",
                    "close": "close_2",
                }
            ),
            on=["timestamp", "from_asset_id_2", "to_asset_id_2"],
            how="left",
        )
        chunk_data[
            [
                "close_1",
                "close_2",
                "from_asset_name_1",
                "to_asset_name_1",
                "from_asset_name_2",
                "to_asset_name_2",
            ]
        ] = (
            chunk_data.sort_values(by="timestamp")
            .groupby(
                [
                    "from_asset_id_1",
                    "to_asset_id_1",
                    "from_asset_id_2",
                    "to_asset_id_2",
                ],
            )[
                [
                    "close_1",
                    "close_2",
                    "from_asset_name_1",
                    "to_asset_name_1",
                    "from_asset_name_2",
                    "to_asset_name_2",
                ]
            ]
            .ffill()
        )
        chunk_data.dropna(inplace=True)
        return chunk_data

    # Split the calculation frame into chunks to process in parallel.
    chunks = []
    chunk_size = 10
    chunk_indices = list(range(0, len(currency_combinations), chunk_size))
    chunk_combinations_list = [
        currency_combinations[i : i + chunk_size] for i in chunk_indices
    ]
    with concurrent.futures.ThreadPoolExecutor() as executor:
        results = list(
            tqdm(
                executor.map(process_chunk, chunk_combinations_list),
                total=len(chunk_combinations_list),
            )
        )
        chunks.extend(results)

    # Combine all chunks into a single Dask DataFrame
    dask_chunks = [dd.from_pandas(chunk, npartitions=2) for chunk in chunks]

    # Set the optimal partition size for your use case.
    calculation_frame = dd.concat(dask_chunks)
    calculation_frame = calculation_frame.repartition(partition_size="100MB")

    return calculation_frame


# Use the Dask-optimized function
calculation_frame = create_calculation_frame_dask(timeframe, currency_id_combinations)
print(f"The calculation frame has {len(calculation_frame)} rows.")

# Get the first 1000 rows of the calculation frame.
calculation_frame.head(1000)

Now that we have a calculation frame, we can start computing our rolling cointegration which will be starting point for our algorithm. This is because for our algorithm to work the pair must be cointegrated over some rolling period previous to the current timestamp.

For cointegration, we will perform a rolling regression of the following equation:

$$
X_t^1 = \alpha + \beta X_t^2 + \epsilon_t
$$

Where $X_t^1$ is the price of the first currency, $X_t^2$ is the price of the second currency, $\alpha$ is the y-intercept of the linear regression, $\beta$ is the slope, and $\epsilon_t$ is the time dependant error of the regression. For our cointegration test, we will test if $\epsilon_t$ is stationary. For calculating stationarity of $\epsilon_t$, we will use the Augmented Dickey-Fuller (ADF) unit root test on the residuals. For now, I will use the function `adfuller` from `statsmodels`.

First we will compute the rolling OLS of each pair combination.

In [ ]:
ROLLING_OLS_COLUMNS = {
    "alpha": float,
    "beta": float,
    "p_value": float,
    "spread_mean": float,
    "spread_std": float,
}


# Define the rolling function that returns only p-value
def rolling_adf_p_value(
    series_1: np.ndarray, series_2: np.ndarray, alpha: float, beta: float
) -> np.float64:
    """Calculate ADF p-value for a rolling window"""
    if len(series_1) != len(series_2):
        raise ValueError(
            "The parameters series_1 and series_2 must be the same length."
        )
    if len(series_1) < 2:
        return np.nan
    try:
        # Return only the p-value (index 1)
        return adfuller(series_2 - alpha - beta * series_1)[1]
    except:
        return np.nan


def rolling_ols(df: pd.DataFrame, window: int = 100, step: int = 1) -> pd.DataFrame:
    # If the window is larger than the number of rows, return a DataFrame with None values.
    if len(df) < window:
        return pd.DataFrame(
            {
                "timestamp": df["timestamp"],
                "alpha": [np.nan] * len(df),
                "beta": [np.nan] * len(df),
                "p_value": [np.nan] * len(df),
            }
        ).set_index(["timestamp"])

    # Compute the rolling OLS.
    X = df["close_1"]
    y = df["close_2"]
    X = sm.add_constant(X)
    result = (
        RollingOLS(y, X, window=window)
        .fit()
        .params.rename(columns={"const": "alpha", "close_1": "beta"})
    )
    result["timestamp"] = df["timestamp"]
    result["close_1"] = df["close_1"]
    result["close_2"] = df["close_2"]

    # Compute the ADF unit root test on the residuals using a for loop.
    p_values = np.zeros(len(result)) * np.nan
    spread_means = np.zeros(len(result)) * np.nan
    spread_stds = np.zeros(len(result)) * np.nan
    alpha_arr = result["alpha"].values
    beta_arr = result["beta"].values
    close_1_arr = result["close_1"].values
    close_2_arr = result["close_2"].values
    for i in range(window - 1, len(result), step):
        window_alpha = alpha_arr[i]
        window_beta = beta_arr[i]
        window_close_1 = close_1_arr[i - window + 1 : i + 1]
        window_close_2 = close_2_arr[i - window + 1 : i + 1]
        window_spread = window_close_2 - window_alpha - window_beta * window_close_1
        spread_mean = window_spread.mean()
        spread_std = window_spread.std()
        p_val = rolling_adf_p_value(
            window_close_1, window_close_2, window_alpha, window_beta
        )
        p_values[i] = p_val
        spread_means[i] = spread_mean
        spread_stds[i] = spread_std
    result["p_value"] = p_values
    result["spread_mean"] = spread_means
    result["spread_std"] = spread_stds

    # Forward fill the results.
    result["p_value"] = result["p_value"].ffill()
    result["spread_mean"] = result["spread_mean"].ffill()
    result["spread_std"] = result["spread_std"].ffill()

    # Only output the same columns as the input.
    output = result.set_index(["timestamp"])[list(ROLLING_OLS_COLUMNS.keys())]

    return output

In [ ]:
calculation_frame_finished = (
    calculation_frame.merge(
        calculation_frame.groupby(
            [
                "from_asset_id_1",
                "to_asset_id_1",
                "from_asset_id_2",
                "to_asset_id_2",
            ]
        )
        .apply(
            lambda x: rolling_ols(
                x,
                window=lookback_window,
                step=lookback_step,
            ),
            meta=ROLLING_OLS_COLUMNS,
        )
        .reset_index()
        .compute(),
        on=["timestamp", "from_asset_id_1", "to_asset_id_1", "from_asset_id_2", "to_asset_id_2"],
        how="left",
    )
)

In [ ]:
# Calculate the spread.
calculation_frame_finished["spread"] = (
    calculation_frame_finished["close_2"]
    - calculation_frame_finished["alpha"]
    - calculation_frame_finished["beta"] * calculation_frame_finished["close_1"]
)

# Calculate the z-score of the spread.
calculation_frame_finished["spread_z_score"] = (
    calculation_frame_finished["spread"] - calculation_frame_finished["spread_mean"]
) / calculation_frame_finished["spread_std"]

In [ ]:
opportunities = calculation_frame_finished.loc[(calculation_frame_finished["p_value"] < 0.01) & (calculation_frame_finished["spread_z_score"].abs() > 3)].sort_values(by=["timestamp"]).compute()
opportunities

In [ ]:
opportunity_pairs = opportunities[["to_asset_name_1", "to_asset_name_2"]].drop_duplicates().apply(tuple, axis=1).tolist()
opportunity_pairs


In [ ]:
# Limit to max 10 opportunity pairs
max_pairs = min(10, len(opportunity_pairs))
pairs_to_plot = opportunity_pairs[:max_pairs]

# Create 5x2 grid (or adjust based on number of pairs)
rows = 5
cols = 2
fig, axs = plt.subplots(rows, cols, figsize=(15, 20))

# Flatten the axes array for easier indexing
if max_pairs == 1:
    axs = [axs]
else:
    axs = axs.flatten()

# Plot each pair
for i, (asset_1, asset_2) in tqdm(enumerate(pairs_to_plot), total=len(pairs_to_plot)):
    if i >= max_pairs:
        break
        
    ax = axs[i]
    
    # Get data for this pair
    pair_data = calculation_frame_finished.loc[(calculation_frame_finished["to_asset_name_1"] == asset_1) & (calculation_frame_finished["to_asset_name_2"] == asset_2)].sort_values(by=["timestamp"]).compute()
    
    # Plot the data
    ax.plot(pair_data['timestamp'], pair_data['spread'], label='Spread')
    
    # Add horizontal lines for mean and std
    spread_mean = pair_data['spread_mean'].mean()
    spread_std = pair_data['spread_std'].mean()
    
    ax.plot(pair_data['timestamp'], pair_data['spread_mean'], color='red', linestyle='--', alpha=0.5, label='Mean')
    ax.plot(pair_data['timestamp'], pair_data['spread_mean'] + 2*pair_data['spread_std'], color='orange', linestyle=':', alpha=0.75, label='+2σ')
    ax.plot(pair_data['timestamp'], pair_data['spread_mean'] - 2*pair_data['spread_std'], color='orange', linestyle=':', alpha=0.75, label='-2σ')
    
    # Customize the plot
    ax.set_title(f'Pair: {asset_1} / {asset_2}')
    ax.set_xlabel('Time')
    ax.set_ylabel('Spread')
    ax.set_ylim(spread_mean - 10*spread_std, spread_mean + 10*spread_std)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Rotate x-axis labels for better readability
    ax.tick_params(axis='x', rotation=45)

# Hide unused subplots
for i in range(max_pairs, rows * cols):
    axs[i].set_visible(False)

# Adjust layout
plt.tight_layout()
plt.show()